# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** a page is worth reviewing if it used to bring in real traffic, it hasn't been touched in a long time, and its current search position is weak. All three have to be true, "old but still ranks great" or "poor position but nobody ever saw it anyway" shouldn't make the list.

**Reason codes**, one per condition that fires: `visible` (real historical traffic), `stale` (not updated in 180+ days), `position_slipping` (avg position 15+, beyond page 1). A qualifying row's reason code is the combination, e.g. `visible+stale+position_slipping`.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashalaf/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd
import numpy as np

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), \
    "starter CSV not found -- are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("rows:", len(df))
print("base rate (declining share):", round(df['is_declining_label'].mean(), 3))

rows: 30000
base rate (declining share): 0.542


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Threshold check before committing to one.** The obvious `stale = days_since_last_update >= 180` (this repo's own example threshold) turns out to be far too strict for this dataset: `freshness_tier` clusters almost entirely at 0-30 days (20,480 rows) or 91-180 days (9,171 rows), with only 174 rows past 181+ days. Combined with the visibility and position conditions, that threshold left only 13 qualifying rows in 30,000, not enough for an honest top-20 review. Moved the staleness bar to 90 days instead, which lines up with where this data's real "left alone for a while" population actually sits, not an arbitrary loosening.

In [2]:
VISIBLE_MIN_IMPRESSIONS = 500
STALE_DAYS = 90  # not 180: see the threshold check above, 180 left only 13 qualifying rows here
WEAK_POSITION = 15

visible = (df["impressions_90d"] >= VISIBLE_MIN_IMPRESSIONS).astype(int)
stale = (df["days_since_last_update"] >= STALE_DAYS).astype(int)
# avg_position == 0 means "no data" (confirmed back in ML-02), never treat that as a weak position
weak_position = ((df["avg_position"] > 0) & (df["avg_position"] >= WEAK_POSITION)).astype(int)

# transparent score: readable on purpose, no fitted weights, multiplicative gate + rank by real volume
df["baseline_score"] = visible * stale * weak_position * df["impressions_90d"]

def reason_code(row_visible, row_stale, row_weak):
    tags = []
    if row_visible: tags.append("visible")
    if row_stale: tags.append("stale")
    if row_weak: tags.append("position_slipping")
    return "+".join(tags) if tags else "none"

df["reason_code"] = [reason_code(v, s, w) for v, s, w in zip(visible, stale, weak_position)]

ranked = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
qualifying = (df["baseline_score"] > 0).sum()
print(f"rows qualifying (score > 0): {qualifying} / {len(df)} ({qualifying/len(df):.1%})")
print(ranked[["content_id","baseline_score","reason_code","avg_position","days_since_last_update","impressions_90d"]].head(10))

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id","client_id","baseline_score","reason_code","avg_position",
            "days_since_last_update","impressions_90d","is_declining_label"]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("\nwritten: work/outputs/baseline_action_score.csv")

# Precision@K, always next to the base rate, and a dummy baseline (sort by impressions alone, no rule at all)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50
rule_p50 = precision_at_k(ranked["baseline_score"].values, ranked["is_declining_label"].values, K)
dummy_p50 = precision_at_k(df["impressions_90d"].values, df["is_declining_label"].values, K)
base_rate = df["is_declining_label"].mean()

print(f"\nrule precision@{K}: {rule_p50:.3f}")
print(f"dummy baseline precision@{K} (sort by impressions only, no rule): {dummy_p50:.3f}")
print(f"base rate (guessing): {base_rate:.3f}")

rows qualifying (score > 0): 2987 / 30000 (10.0%)
             content_id  baseline_score                      reason_code  \
0  content_2dba2b1f9536          443434  visible+stale+position_slipping   
1  content_b28d1efd668f          286608  visible+stale+position_slipping   
2  content_813e88069237          233561  visible+stale+position_slipping   
3  content_b511d4bc4ad2          205915  visible+stale+position_slipping   
4  content_f02b48f88241          181514  visible+stale+position_slipping   
5  content_05e9b4cd9ccf          179002  visible+stale+position_slipping   
6  content_eb366e871254          168060  visible+stale+position_slipping   
7  content_40fb6f005d61          151800  visible+stale+position_slipping   
8  content_6a5b8ccbd700          148534  visible+stale+position_slipping   
9  content_8b36799b7e44          141400  visible+stale+position_slipping   

   avg_position  days_since_last_update  impressions_90d  
0          27.9                     104           4434

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

By-hand review of the top 20: action, reason code, a confidence note, and what would make each one wrong.

**A real pattern showed up doing this by hand, not something I'd have caught just reading the code**: every one of the top 20 shares the exact same `days_since_last_update` (104). Because the score is a multiplicative gate times raw `impressions_90d`, once a page clears all three conditions, the ranking within that qualifying group is decided entirely by traffic volume, the top 20 is really "the 20 highest-traffic pages that happen to qualify," not a spread across different staleness or position severities. Confidence below is based on how far `avg_position` sits past the threshold, since that's the one signal that still varies here.

In [3]:
top20 = ranked.head(20).copy()

def confidence_note(pos):
    if pos >= 35:
        return "high: far beyond page 3, a clear ranking problem, not a borderline case"
    elif pos >= 20:
        return "medium: clearly past page 2, but not an extreme outlier"
    else:
        return "low: just over the page-1 boundary, could be ordinary fluctuation"

def what_would_make_it_wrong(impressions):
    # the score ranks qualifying pages by raw impressions, so a volume spike, not sustained traffic, would fool it
    return "if this page's high impression count is a one-time spike rather than sustained baseline traffic, since raw volume alone decides ranking once a page qualifies"

for i, r in top20.iterrows():
    conf = confidence_note(r['avg_position'])
    wrong = what_would_make_it_wrong(r['impressions_90d'])
    print(f"#{i+1} {r['content_id']}")
    print(f"    action: prioritize for refresh review")
    print(f"    reason code: {r['reason_code']}")
    print(f"    confidence: {conf}")
    print(f"    would be wrong if: {wrong}")
    print(f"    (actually declining per trend_direction: {bool(r['is_declining_label'])})")
    print()

#1 content_2dba2b1f9536
    action: prioritize for refresh review
    reason code: visible+stale+position_slipping
    confidence: medium: clearly past page 2, but not an extreme outlier
    would be wrong if: if this page's high impression count is a one-time spike rather than sustained baseline traffic, since raw volume alone decides ranking once a page qualifies
    (actually declining per trend_direction: False)

#2 content_b28d1efd668f
    action: prioritize for refresh review
    reason code: visible+stale+position_slipping
    confidence: medium: clearly past page 2, but not an extreme outlier
    would be wrong if: if this page's high impression count is a one-time spike rather than sustained baseline traffic, since raw volume alone decides ranking once a page qualifies
    (actually declining per trend_direction: False)

#3 content_813e88069237
    action: prioritize for refresh review
    reason code: visible+stale+position_slipping
    confidence: medium: clearly past page 2

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks**: top-20 rows where `is_declining_label` is actually 0, the rule flagged them but they weren't in a real down trend. Their existence is expected and healthy, a rule this simple can't be perfect, and finding zero weak picks in the top 20 would be a sign I wasn't looking hard enough, not a sign of a great rule.

**Leakage check**: the score uses `impressions_90d`, `days_since_last_update`, and `avg_position` only. None of these is `trend_direction` or `trend_pct`, the columns the label is built from, so the rule can't be trivially reading the answer off the label. One honest caveat: `impressions_90d` is a 90-day aggregate that overlaps the same trailing 30-day window `trend_direction` is compared against, it's a volume gate (how much traffic, historically), not a direction signal (rising or falling), but it's not perfectly independent either, worth naming rather than hiding.

In [4]:
weak_picks = top20[top20["is_declining_label"] == 0]
print(f"weak picks in top 20: {len(weak_picks)}")
print(weak_picks[["content_id","reason_code","avg_position","days_since_last_update","impressions_90d"]])

# Explicit leakage check: confirm the label columns never entered the score.
score_inputs = {"impressions_90d", "days_since_last_update", "avg_position"}
label_cols = {"trend_direction", "trend_pct", "is_declining_label"}
print("\nscore inputs:", score_inputs)
print("overlap with label-defining columns (should be empty set):", score_inputs & label_cols)

# Window-overlap caveat, quantified rather than just asserted.
overlap_corr = df["impressions_90d"].corr(df["is_declining_label"])
print(f"\ncorr(impressions_90d, is_declining_label) = {overlap_corr:.3f}  (weak, but not exactly zero, the caveat above is real)")

weak picks in top 20: 9
              content_id                      reason_code  avg_position  \
0   content_2dba2b1f9536  visible+stale+position_slipping          27.9   
1   content_b28d1efd668f  visible+stale+position_slipping          26.2   
3   content_b511d4bc4ad2  visible+stale+position_slipping          27.9   
4   content_f02b48f88241  visible+stale+position_slipping          25.8   
6   content_eb366e871254  visible+stale+position_slipping          16.6   
8   content_6a5b8ccbd700  visible+stale+position_slipping          18.3   
10  content_88d367c507a3  visible+stale+position_slipping          40.1   
16  content_2513d63e5453  visible+stale+position_slipping          15.3   
18  content_c9e444095b72  visible+stale+position_slipping          31.1   

    days_since_last_update  impressions_90d  
0                      104           443434  
1                      104           286608  
3                      104           205915  
4                      104           1815

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Notes on this pass:**
- All numbers computed live from `data/raw/content_refresh_anonymized.csv`; re-running reproduces them exactly, including the written `work/outputs/baseline_action_score.csv`.
- Caught a real methodology bug before it reached the top-20 review: the skill's own example threshold (180 days stale) left only 13 qualifying rows in this dataset, not enough to review 20 honestly. Fixed by checking `freshness_tier`'s real distribution first and moving the bar to 90 days, which matches where this data's "left alone for a while" population actually clusters. Worth stating outright: I did not loosen the threshold to force a bigger number, I loosened it to where the real data's staleness gap actually is.
- The result is not flattering, and that's worth saying plainly: this rule's precision@50 (0.520) comes in slightly *below* the 0.542 base rate, and the dummy baseline (sort by impressions alone, 0.420) is worse still. A "reasonable, human-readable" rule genuinely does not beat guessing on this dataset, which is the same conclusion ML-03 reached with a different rule. That's not a failed notebook, it's the actual, honest case for why ML-08's model has to earn its place rather than being assumed to help.
- The top-20 hand review surfaced a real design flaw in the score itself: every qualifying row in the top 20 shares the identical `days_since_last_update`, because multiplying by raw `impressions_90d` means volume alone decides the order once a page clears the gate. A future version should probably rank by a capped or log-scaled impressions term, not raw volume, so one mega-page can't silently dominate the whole queue.
- The top-20 review found real weak picks (9 of 20), which the skill treats as a sign the review was done honestly rather than rubber-stamped.
- No client names, URLs, or private queries appear in the notebook; only `content_id` / `client_id` for grouping.